# Axis — CODE-15 deep-backbone pretraining (Colab)

Pretrains `ECGResNetDeep1D` on CODE-15% and produces `backbone.pt` (~27 MB) to
bring back to the laptop for the local fine-tune.

## Antes de empezar (importante)
**`Entorno de ejecución` → `Cambiar tipo de entorno` → `GPU` → Guardar.**

Luego ejecuta las celdas **en orden** (botón ▶ a la izquierda de cada una).
Cada celda es independiente: si una falla puedes re-ejecutarla sin repetir las
anteriores.

In [ ]:
#@title 1) Comprobar GPU y disco
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'SIN GPU -> cambia el entorno a GPU'
!echo 'Disco libre:'; df -h / | tail -1

In [ ]:
#@title 2) Descargar el codigo (repo publico)
%cd /content
!rm -rf /content/Heartcheck
!git clone -q https://github.com/Davmunrey/Heartcheck
%cd /content/Heartcheck
!git log --oneline -1

In [ ]:
#@title 3) Instalar dependencias (aria2 + torch/numpy/h5py + heartscan_ml)
!command -v aria2c >/dev/null 2>&1 || (apt-get -qq update && apt-get -qq install -y aria2)
!pip install -q numpy h5py
!pip install -q --no-deps -e ml/
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
#@title 4) Descargar CODE-15 (12 partes ~= 230k ECGs, ~30 GB, rapido con aria2)
#@markdown Sube PARTS si tienes mas disco (max 18); bajalo si se llena.
PARTS = 12  #@param {type:"integer"}
import os
os.environ['CODE15_ROOT'] = '/content/code_15pct'
!DEST=/content/code_15pct CODE15_PARTS=$PARTS ARIA_CONN=16 bash scripts/download_code15.sh
!echo '--- partes descargadas ---'; ls /content/code_15pct/*.hdf5 2>/dev/null | wc -l
!df -h / | tail -1

In [ ]:
#@title 5) Pre-entrenar el backbone (entrena con las partes que haya en disco)
#@markdown EPOCHS=10 recomendado; baja a 5 si la T4 va lenta. Baja BATCH si hay OOM.
EPOCHS = 10  #@param {type:"integer"}
BATCH = 128  #@param {type:"integer"}
!python -m ml.training.pretrain_code15 \
    --data-root /content/code_15pct --out runs/cloud/code15_pretrain \
    --epochs $EPOCHS --batch-size $BATCH --lr 1e-3 --workers 8

In [ ]:
#@title 6) Descargar el backbone al portatil (solo ~27 MB)
from google.colab import files
files.download('/content/Heartcheck/runs/cloud/code15_pretrain/backbone.pt')

## Despues, en el portatil (fine-tune local)
```bash
cd ~/dev/Heartcheck
mkdir -p runs/local/code15_pretrain
mv ~/Downloads/backbone.pt runs/local/code15_pretrain/backbone.pt
PRETRAIN_OUT=runs/local/code15_pretrain scripts/pretrain_finetune_code15.sh
```
Se salta el pre-entreno (ya tiene el backbone) y fine-tunea las 5 superclases
sobre el blend local. Luego evalua vs el champion **0.608** y promociona solo si
gana. Detalles en `docs/CLOUD_PRETRAIN.md`.

### Si Colab se queda sin disco en la celda 4
Baja `PARTS` (p.ej. 8) y re-ejecuta la celda 4 — el pre-entreno (celda 5) usa
automaticamente solo las partes presentes. 8-12 partes ya son un corpus enorme.